# Anchor Assignments

Method Overview:
1. Preprocessing each image using green channel + CLAHE (Contrast Limited Adaptive Histogram Equalization) + resizing
2. Extract local features using SIFT
3. Match descriptors with Lowe's ratio test
4. Fit a homography with RANSAC
5. Score each anchor-test pair using:
   - number of inliers
   - inlier ratio
   - reprojection error
   - vessel-strucutre (Frangi) similarity
6. Assign each test image to the anchor with the best score



## Kernel Check

In [662]:
#WHAT IS NOT DETERMINISTIC ABOUT THIS CLASS -- homography --> affine
import sys
print(sys.executable)

import cv2
print(cv2.__version__)

/opt/miniconda3/envs/mia/bin/python
4.13.0


## Imports & Requirements

In [663]:
#!/usr/bin/env python3
"""
Requirements to install:
    pip install opencv-python numpy pandas
    pip install scikit-image
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from skimage.filters import frangi
import numpy as np

#Reduce randomness in OpenCV/NumPy dependent steps
cv2.setRNGSeed(0)
np.random.seed(0)


VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

## Argument Parsing & Handling Image Files

In [664]:
# Parse inputs to be given in main method
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Assign each test fundus image to an anchor image.")
    parser.add_argument("--anchors_dir", type=str, required=True, help="Directory containing anchor images")
    parser.add_argument("--tests_dir", type=str, required=True, help="Directory containing test images")
    parser.add_argument("--output_csv", type=str, required=True, help="Path to save grouping.csv")
    parser.add_argument("--diagnostics_csv", type=str, default=None, help="Optional path to save detailed pairwise scores")
    parser.add_argument("--feature", type=str, default="sift", choices=["sift"], help="Feature extractor to use") # Choices if we want to experiment different feature detectors
    parser.add_argument("--resize", type=int, default=512, help="Resize images to resize x resize before matching")
    parser.add_argument("--ratio_thresh", type=float, default=0.75, help="Lowe ratio test threshold")
    parser.add_argument("--ransac_thresh", type=float, default=5.0, help="RANSAC reprojection threshold in pixels")
    parser.add_argument("--clahe_clip", type=float, default=2.0, help="CLAHE clip limit")
    parser.add_argument("--nfeatures", type=int, default=2000, help="Number of features for SIFT")
    return parser.parse_args()


# Confirm valid image file path
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VALID_EXTENSIONS


# Sort image files
def list_image_files(folder: Path) -> List[Path]:
    files = [p for p in sorted(folder.iterdir()) if is_image_file(p)]
    if not files:
        raise FileNotFoundError(f"No image files found in: {folder}")
    return files


# Read image from disk with OpenCV
def load_image(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Failed to read image: {path}")
    return img

## Pre-processing, Feature Detecting, Matching Feature Descriptors

In [665]:
# Suppress dark border/background
# Return binary retinal mask
def create_fundus_mask(gray: np.ndarray) -> np.ndarray:
    mask = (gray > 20).astype(np.uint8) * 255 # Thresholding
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel) # Morphological Closing
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)  # Morphological Opening
    # New: shrink inward to avoid border keypoints
    erode_kernel = np.ones((15, 15), np.uint8)
    mask = cv2.erode(mask, erode_kernel, iterations=1)
    return mask


# Preprocessing
# Return preprocessed grayscale image (proc_img) and binary mask of relevant (non-background) fundus region (mask)
def preprocess_fundus(img_bgr: np.ndarray, resize_to: int = 512, clahe_clip: float = 2.0) -> Tuple[np.ndarray, np.ndarray]:
    if img_bgr.ndim != 3 or img_bgr.shape[2] != 3:
        raise ValueError("Expected a color BGR image")

    # OpenCV loads BGR; green channel is index 1
    green = img_bgr[:, :, 1]

    # Resize first for consistency
    proc = cv2.resize(green, (resize_to, resize_to), interpolation=cv2.INTER_AREA)

    # CLAHE contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    proc = clahe.apply(proc)

    # Create mask from enhanced image
    mask = create_fundus_mask(proc)

    return proc, mask


# Vessel enhancement using Sobel + CLAHE

def enhance_vessels(img):
    img = img.astype(np.float32) / 255.0                # Convert image to float32 & normalize
    sobelx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)  # Horizontal gradient (highlight vertical edges)
    sobely = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)  # Vertical gradient (highlight horizontal edges)
    vessel = np.sqrt(sobelx**2 + sobely**2)             # Gradient magnitude
    return vessel


# Fangi Enhancement -- LESS SUCCESSFUL
#def enhance_vessels(img):
    # Convert to float in [0, 1]
    img = img.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0

    # True Frangi vesselness filter
    vessel = frangi(
        img,
        sigmas=range(1, 6),      # vessel scales to search over
        alpha=0.5,               # blob sensitivity
        beta=0.5,                # plate-like vs line-like sensitivity
        gamma=None,              # use automatic normalization
        black_ridges=True       # vessels are bright after your preprocessing
    )

    return vessel.astype(np.float32)


# Create SIFT feature detector
def make_detector(feature_type: str, nfeatures: int):
    if feature_type == "sift":  # We can try different feature detectors later if wanted
        if not hasattr(cv2, "SIFT_create"): # Safety check to ensure correct cv2 build
            raise RuntimeError("SIFT is not available in your OpenCV build --> Install a build that includes SIFT")
        return cv2.SIFT_create(nfeatures=nfeatures)
    raise ValueError(f"Unsupported feature type: {feature_type}")


# Return key points and computed descriptors around each keypoint given the image and mask
def extract_features(img: np.ndarray, mask: np.ndarray, detector) -> Tuple[List[cv2.KeyPoint], Optional[np.ndarray]]:
    kp, des = detector.detectAndCompute(img, mask)
    return kp, des
    

# Match feature descriptors between 2 images, returns list of reliable feature matches
def match_descriptors(des1: Optional[np.ndarray], des2: Optional[np.ndarray], feature_type: str, ratio_thresh: float) -> List[cv2.DMatch]:
    if des1 is None or des2 is None:  # i.e. No matches if no descriptors
        return []                   

    if len(des1) < 2 or len(des2) < 2: # Too few matches, need two nearest neighbors
        return []

    if feature_type == "sift":
        # FLANN (Fast Library for Approximate Nearest Neighbors) mactcher for float descriptors
        #index_params = dict(algorithm=1, trees=5)  # KD-tree --> K dimensional tree, splitting data along different dimensions to compare points, eliminating searching through regions with bad matches
        #search_params = dict(checks=50)
        #FLANN matcher = cv2.FlannBasedMatcher(index_params, search_params)
        #Brute Force 
        matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck = False)
        knn_matches = matcher.knnMatch(des1, des2, k=2) # Find two nearest neighbors

    else:
        raise ValueError(f"Unsupported feature type: {feature_type}") # Safety check

    good_matches: List[cv2.DMatch] = [] # Initialize list of good matches
    for pair in knn_matches:
        if len(pair) < 2:
            continue
        m, n = pair # m --> best match, n --> second best match
        if m.distance < ratio_thresh * n.distance:  # Lowe's ratio test, keeping matches where best match is significantly better than second best
            good_matches.append(m)

    return good_matches

## Scoring Similarity of Anchor-Test Images & Assigning Anchors

In [666]:
# Compute reprojection error of how given transformation H aligns point sets 1 and 2
def compute_reprojection_error(pts1: np.ndarray, pts2: np.ndarray, A: np.ndarray) -> float:
    if len(pts1) == 0:  # If no points, there is an infinite reprojection error (i.e. meaningless alignment)
        return float("inf")

    # reshape to N x 2
    pts1 = pts1.reshape(-1, 2)
    pts2 = pts2.reshape(-1, 2)


    # affine transform: [x', y'] = A @ [x, y, 1]
    ones = np.ones((pts1.shape[0], 1), dtype=np.float32)
    pts1_h = np.hstack([pts1, ones])           # N x 3
    pts1_proj = (A @ pts1_h.T).T               # N x 2

    err = np.sqrt(np.sum((pts1_proj - pts2) ** 2, axis=1))
    return float(np.mean(err)) if len(err) > 0 else float("inf")


def ransac_score(kp1: List[cv2.KeyPoint], kp2: List[cv2.KeyPoint], matches: List[cv2.DMatch], ransac_thresh: float) -> Dict[str, float]:

    stats = {
        "num_matches": float(len(matches)),
        "num_inliers": 0.0,
        "inlier_ratio": 0.0,
        "mean_error": float("inf"),
    }

    if len(matches) < 3:   # affine needs at least 3 points to estimate an affine transform
        return stats

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])   # shape (N, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])   # shape (N, 2)

    A, mask = cv2.estimateAffine2D(pts1, pts2, method=cv2.RANSAC, ransacReprojThreshold=ransac_thresh)

    if A is None or mask is None:
        return stats

    inlier_mask = mask.ravel().astype(bool)
    num_inliers = int(np.sum(inlier_mask))
    inlier_ratio = num_inliers / max(len(matches), 1)

    if num_inliers > 0:
        pts1_in = pts1[inlier_mask]
        pts2_in = pts2[inlier_mask]
        mean_error = compute_reprojection_error(pts1_in, pts2_in, A)
    else:
        mean_error = float("inf")

    stats["num_inliers"] = float(num_inliers)
    stats["inlier_ratio"] = float(inlier_ratio)
    stats["mean_error"] = float(mean_error)

    return stats


# Similarity score of two images' vessel structures
def vessel_similarity(img1, img2, mask1=None, mask2=None):
    v1 = enhance_vessels(img1)
    v2 = enhance_vessels(img2)

    if mask1 is not None:
        v1 = v1[mask1 > 0]
    else:
        v1 = v1.ravel()

    if mask2 is not None:
        v2 = v2[mask2 > 0]
    else:
        v2 = v2.ravel()

    n = min(len(v1), len(v2))
    v1 = v1[:n]
    v2 = v2[:n]

    if np.std(v1) < 1e-6 or np.std(v2) < 1e-6:
        return 0.0

    return float(np.corrcoef(v1, v2)[0, 1])



# Weigh metrics to compute final similarity score (weights have been & can be further optimized)
def final_pair_score(stats: Dict[str, float], vessel_sim: float) -> float:
    num_matches = stats["num_matches"]
    num_inliers = stats["num_inliers"]
    inlier_ratio = stats["inlier_ratio"]
    mean_error = stats["mean_error"]

    if np.isinf(mean_error) or np.isnan(mean_error):
        mean_error = 1e9

    score = (
        0.03 * num_matches
        + 3.0 * num_inliers
        + 55.0 * inlier_ratio
        - 12.0 * mean_error
        + 20.0 * vessel_sim
    )
    return float(score)


#NEW HELPER
def suppress_optic_disc(proc: np.ndarray, mask: np.ndarray) -> np.ndarray:
    masked = proc.copy()
    masked[mask == 0] = 0

    # Find brightest region inside the fundus as a rough optic disc estimate
    blur = cv2.GaussianBlur(masked, (31, 31), 0)
    _, _, _, max_loc = cv2.minMaxLoc(blur)

    disc_mask = np.ones_like(mask, dtype=np.uint8) * 255
    radius = int(0.08 * proc.shape[0])   # try 6% to 9% of image size
    cv2.circle(disc_mask, max_loc, radius, 0, -1)

    final_mask = cv2.bitwise_and(mask, disc_mask)
    return final_mask


# Cache to locally & temporarily store feature data (path, pre-processed image, mask, keypoints, descriptors)
def build_feature_cache(image_paths: List[Path], detector, resize_to: int, clahe_clip: float) -> Dict[str, Dict[str, object]]:
    cache: Dict[str, Dict[str, object]] = {}

    for path in image_paths:
        img_bgr = load_image(path)

        proc, mask = preprocess_fundus(img_bgr, resize_to=resize_to, clahe_clip=clahe_clip)
        feature_mask = suppress_optic_disc(proc, mask)
        kp, des = extract_features(proc, feature_mask, detector)

        cache[path.name] = {
            "path": path,
            "proc": proc,
            "mask": feature_mask,
            "kp": kp,
            "des": des,
        }
        
        print(f"Cached {path.name}: {len(kp)} keypoints")

    return cache

# Assign a test image to its best anchor and return statistics & diagnostics
def assign_single_test_image(test_name: str, test_entry: Dict[str, object], anchor_cache: Dict[str, Dict[str, object]], feature_type: str, ratio_thresh: float, ransac_thresh: float) -> Tuple[str, Dict[str, float], List[Dict[str, object]]]:
    pairwise_rows: List[Dict[str, object]] = []    # List for anchor-test comparison
    best_anchor_name: Optional[str] = None         # Initialize anchor name
    best_score = -float("inf")                     # Initialize best score (start low)
    best_stats: Optional[Dict[str, float]] = None  # Initialize stats for best match

    # Extract key points, descriptors, and processed image for test image
    kp_test = test_entry["kp"]
    des_test = test_entry["des"]
    test_img = test_entry["proc"]

    # Look through all anchor images
    for anchor_name, anchor_entry in anchor_cache.items():
        # Extract key points, descriptors, and processed image for anchor image
        kp_anchor = anchor_entry["kp"]
        des_anchor = anchor_entry["des"]
        anchor_img = anchor_entry["proc"]

        # Match features between anchor and test
        matches = match_descriptors(des_anchor, des_test, feature_type=feature_type, ratio_thresh=ratio_thresh)

        # Use RANSAC to find num inliers, inlier ratio, reprojection error
        stats = ransac_score(kp_anchor, kp_test, matches, ransac_thresh=ransac_thresh)

        vessel_sim = vessel_similarity(anchor_img, test_img, mask1=anchor_entry["mask"], mask2=test_entry["mask"]) # Gloabl vessel similarity
        score = final_pair_score(stats, vessel_sim)          # Combine metrics into final score


        # Dictionary storing test image, anchor image, score, and all metrics
        row = {
            "test_image": test_name,
            "anchor_image": anchor_name,
            "score": score,
            "num_matches": int(stats["num_matches"]),
            "num_inliers": int(stats["num_inliers"]),
            "inlier_ratio": float(stats["inlier_ratio"]),
            "mean_error": float(stats["mean_error"]),
            "vessel_sim": float(vessel_sim),
        }
        pairwise_rows.append(row) # Add results to list

        # Compare to existing best anchor image and re-assign if better match (i.e. if higher score)
        if score > best_score:
            best_score = score
            best_anchor_name = anchor_name
            best_stats = stats


    pairwise_rows_sorted = sorted(pairwise_rows, key=lambda r: r["score"], reverse=True)

    
    if best_anchor_name is None or best_stats is None: # Failure case if no valid assignment
        raise RuntimeError(f"Failed to assign anchor for test image: {test_name}")

    return best_anchor_name, best_stats, pairwise_rows

## Main Method --> Assigning Anchors for All Test Images!

In [667]:
def main() -> None:
    # Change paths for different devices / when we get testing data
    anchors_dir = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/anchor_images")
    tests_dir = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/test_images")
    output_csv = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/grouping_results.csv")
    diagnostics_csv = Path("/Users/nataliesmith/Desktop/MIA Project 1/example test/diagnostics_results.csv")

    # Params chosen to maximize performance
    feature = "sift"
    resize = 768
    ratio_thresh = 0.40
    ransac_thresh = 3.0
    clahe_clip = 2.0
    nfeatures = 4000

    # Safety check
    if not anchors_dir.exists():
        raise FileNotFoundError(f"Anchors directory does not exist: {anchors_dir}")
    if not tests_dir.exists():
        raise FileNotFoundError(f"Tests directory does not exist: {tests_dir}")

    anchor_paths = list_image_files(anchors_dir)
    test_paths = list_image_files(tests_dir)

    # Confirm correct number of anchor and test images
    print(f"Found {len(anchor_paths)} anchor images")
    print(f"Found {len(test_paths)} test images")

    # Feature detector w chosen parameters
    detector = make_detector(feature, nfeatures)

    # Chache's for anchor and test images
    print("\nBuilding anchor cache...")
    anchor_cache = build_feature_cache(anchor_paths, detector=detector, resize_to=resize, clahe_clip=clahe_clip)

    print("\nBuilding test cache...")
    test_cache = build_feature_cache(test_paths, detector=detector, resize_to=resize, clahe_clip=clahe_clip)

    grouping_rows = []
    diagnostics_rows = []

    # Anchor assignment
    print("\nAssigning anchors...")
    for test_name, test_entry in test_cache.items():
        assigned_anchor, stats, pairwise_rows = assign_single_test_image(test_name=test_name, test_entry=test_entry, anchor_cache=anchor_cache, feature_type=feature, ratio_thresh=ratio_thresh, ransac_thresh=ransac_thresh)

        print(
            f"{test_name} -> {assigned_anchor} | "
            f"inliers={int(stats['num_inliers'])}, "
            f"inlier_ratio={stats['inlier_ratio']:.3f}, "
            f"mean_error={stats['mean_error']:.3f}"
        )

        grouping_rows.append({"test_image": test_name,"anchor_image": assigned_anchor,})
        diagnostics_rows.extend(pairwise_rows)


    output_csv.parent.mkdir(parents=True, exist_ok=True)  # Ensure output directory exists
    grouping_df = pd.DataFrame(grouping_rows)             # Convert results to pandas DataFrame
    grouping_df.to_csv(output_csv, index=False)           # Save as .csv
    print(f"\nSaved grouping CSV to: {output_csv}")       # Print location of csv

    diagnostics_csv.parent.mkdir(parents=True, exist_ok=True)
    diagnostics_df = pd.DataFrame(diagnostics_rows)
    diagnostics_df.sort_values(["test_image", "score"], ascending=[True, False], inplace=True)  # Sort by test_image and score (highest first)
    diagnostics_df.to_csv(diagnostics_csv, index=False)
    print(f"Saved diagnostics CSV to: {diagnostics_csv}")



    # Misalignment Testing
    # Ground truth mapping: test_image -> correct anchor_image
    ground_truth = {
        "test_01.tiff": "anchor_04.tiff",
        "test_02.tiff": "anchor_02.tiff",
        "test_03.tiff": "anchor_01.tiff",
        "test_04.tiff": "anchor_01.tiff",
        "test_05.tiff": "anchor_04.tiff",
        "test_06.tiff": "anchor_05.tiff",
        "test_07.tiff": "anchor_05.tiff",
        "test_08.tiff": "anchor_05.tiff",
        "test_09.tiff": "anchor_04.tiff",
        "test_10.tiff": "anchor_02.tiff",
        "test_11.tiff": "anchor_03.tiff",
        "test_12.tiff": "anchor_05.tiff",
        "test_13.tiff": "anchor_04.tiff",
        "test_14.tiff": "anchor_01.tiff",
        "test_15.tiff": "anchor_03.tiff",
        "test_16.tiff": "anchor_05.tiff",
        "test_17.tiff": "anchor_02.tiff",
        "test_18.tiff": "anchor_04.tiff",
        "test_19.tiff": "anchor_05.tiff",
        "test_20.tiff": "anchor_03.tiff",
        "test_21.tiff": "anchor_03.tiff",
        "test_22.tiff": "anchor_04.tiff",
        "test_23.tiff": "anchor_03.tiff",
        "test_24.tiff": "anchor_05.tiff",
        "test_25.tiff": "anchor_03.tiff",
        
    }
    
    num_misassignments = 0;

    for test_name, test_entry in test_cache.items():
        assigned_anchor, stats, pairwise_rows = assign_single_test_image(
        test_name, test_entry, anchor_cache,
        feature_type=feature,
        ratio_thresh=ratio_thresh,
        ransac_thresh=ransac_thresh
        )

        # Check against ground truth
        if test_name in ground_truth:
            if assigned_anchor != ground_truth[test_name]:
                num_misassignments += 1

    print(f"\nNumber of misassignments: {num_misassignments} / {len(test_cache)}")
    if test_name in ground_truth:
        if assigned_anchor != ground_truth[test_name]:
            num_misassignments += 1
            print(f"Mismatch: {test_name} -> predicted {assigned_anchor}, true {ground_truth[test_name]}")


if __name__ == "__main__":
    main()

Found 5 anchor images
Found 25 test images

Building anchor cache...
Cached anchor_01.tiff: 1869 keypoints
Cached anchor_02.tiff: 439 keypoints
Cached anchor_03.tiff: 983 keypoints
Cached anchor_04.tiff: 865 keypoints
Cached anchor_05.tiff: 857 keypoints

Building test cache...
Cached test_01.tiff: 104 keypoints
Cached test_02.tiff: 437 keypoints
Cached test_03.tiff: 3245 keypoints
Cached test_04.tiff: 1730 keypoints
Cached test_05.tiff: 528 keypoints
Cached test_06.tiff: 460 keypoints
Cached test_07.tiff: 841 keypoints
Cached test_08.tiff: 3586 keypoints
Cached test_09.tiff: 526 keypoints
Cached test_10.tiff: 1411 keypoints
Cached test_11.tiff: 986 keypoints
Cached test_12.tiff: 1971 keypoints
Cached test_13.tiff: 1709 keypoints
Cached test_14.tiff: 1051 keypoints
Cached test_15.tiff: 945 keypoints
Cached test_16.tiff: 876 keypoints
Cached test_17.tiff: 340 keypoints
Cached test_18.tiff: 3574 keypoints
Cached test_19.tiff: 865 keypoints
Cached test_20.tiff: 446 keypoints
Cached test_2